## Initialization

In [ ]:
# Imports
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
from data_processing import paths
from data_processing.experiment_data_keys import (ExperimentDataKey,
                                                  ExperimentNeutronData)
from data_processing.helpers import stop

In [ ]:
CalibrationKey = Literal[ExperimentDataKey.CAEN_CALIBRATION, ExperimentDataKey.NEW_CALIBRATION]
NasaBorderKey = Literal[ExperimentDataKey.NASA_BORDERS, ExperimentDataKey.NASA_BORDERS_RECALC]

## Experiment ID Input

In [ ]:
experiment_ids = [
    "TB-twentyfive_again_threshold",
    "TB-twohundred_fifty_again_threshold",
]

In [ ]:
experiment_neutron_data: ExperimentNeutronData = {
    exp_id: {}
    for exp_id in experiment_ids
}

## Data Loading and Initial Processing

In [ ]:
# Data Loading
for exp_id, exp_data in experiment_neutron_data.items():
    exp_base_path = paths.get_exp_root(exp_id)
    spectra_folder = exp_base_path / "processed_data/unfiltered/spectra"
    time_spectrum_path = spectra_folder / "time_spectrum_0.parquet"
    df = pd.read_parquet(time_spectrum_path)
    print(df.head(200))
    exp_data["time_spectrum"] = df

## Figure Base Data

In [ ]:
dl_folder = Path.home() / "Downloads" / "neutron_detection_paper"
dl_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def get_figure_data(merge_count: int | None = 10):
    """
    Contains histogram data for neutron detector noise
    The histogram shows the time interval distribution (in us)
    The index is the bin start value
    The series values are the bin widths and counts
    """
    name_map = {
        "TB-twentyfive_again_threshold": 25,
        "TB-twohundred_fifty_threshold": 250,
        "TB-twohundred_fifty_again_threshold": 250,
    }
    
    all_exp_dfs = {}
    for exp_id, exp_data in experiment_neutron_data.items():
        name = name_map[exp_id]
        spectrum_df = exp_data["time_spectrum"]
        bin_starts = spectrum_df["bin_time"].to_numpy()
        bin_counts = spectrum_df["count"].to_numpy()
        if merge_count is not None and merge_count > 1:
            # select every merge_count element of bin_starts
            indices = np.arange(0, bin_counts.size, merge_count)
            bin_starts = bin_starts[indices]
            # sum every merge_count slice of bin_counts
            bin_counts = np.add.reduceat(bin_counts, indices)
        bin_widths = bin_starts[1:] - bin_starts[:-1]
        suffix = [bin_widths[-1]]
        bin_widths = np.concatenate((bin_widths, suffix))

        df = pd.DataFrame(
            data={"width": bin_widths, "counts": bin_counts},
            index=bin_starts
        )
        all_exp_dfs[name] = df
    df = pd.concat(all_exp_dfs, axis=1)
    return df

## Plotting

### Plot Style Constants

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial'] + plt.rcParams['font.sans-serif']
fontsize = 20

In [ ]:
bg_blue = "#4c94ff"
bg_red = "#f54336"
bg_grey = "#9e9e9e"
bg_bluegrey = "#8a9fb8"
bg_green = "#21d894"

### Plot Functions 

In [ ]:
def plot_figure(ax: mpl.axes.Axes):
    zorders = {
        25: 0,
        250: 0,
    }
    colors = {
        25: bg_blue,
        250: bg_red,
    }
    alpha = 1
    merge_count = 10

    df = get_figure_data(merge_count)

    for threshold_name, t_threshold_df in df.T.groupby(level=0, sort=False):
        zorder_mod = zorders[threshold_name]
        color = colors[threshold_name]
        threshold_df = t_threshold_df.droplevel(0).T
        ax.bar(
            threshold_df.index,
            threshold_df["counts"],
            width=threshold_df["width"],
            align="edge",
            zorder=5+zorder_mod,
            color=color,
            alpha=alpha,
            lw=0
        )
    ax.set_xlim(0, 1000000)
    ax.set_ylim(0, 25 * merge_count)
    ax.tick_params(labelsize=fontsize)
    ax.set_xlabel(r"Time interval ($\mu$s)", fontsize=fontsize)
    ax.set_ylabel("Counts", fontsize=fontsize)
    ax.xaxis.set_major_formatter(lambda x, _: f"{x / 1000:.0f}")

### Plot Creation

In [ ]:
fig_folder = dl_folder / "figures"
fig_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 8),
    dpi=600,
    layout="constrained"
)
plot_figure(ax)
fig.savefig(fig_folder / "si_fig_3b.png", dpi=fig.dpi, bbox_inches="tight")

In [ ]:
input("Processing done, hit Enter to finish")
stop()

## Base Data Export

In [ ]:
excel_folder = dl_folder / "excel"
excel_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
def export_figure_data():
    df = get_figure_data()
    df = df.rename(columns={"width": "Bin width (us)", "counts": "Counts"})
    df.to_excel(
        excel_folder / "si_figure_3b.xlsx",
        index_label="Time interval (us)"
    )

In [ ]:
export_figure_data()

In [ ]:
input("Processing done, hit Enter to finish")
stop()